# 08 — Cross-generator zero-shot evaluation

Generates Kazakh text with a generator unseen during training, pairs it with the human
documents of the held-out test split, and scores the result under the operational binary
mapping {AI-generated, AI-obfuscated} → machine.

- Input: `data/cleaned/*.json`, weights from notebook 09
- Output: `results_r2_crossgen/table_crossgenerator.csv`, `ood_generations.jsonl`
- Generator: `mistralai/Mistral-7B-Instruct-v0.3`
- Runtime: ~1 h on an A100, resumable
- Reported in: paper Section VI-G

Two disclosures carried into the paper: the Kazakh-native candidates (KazLLM, Sherkala) are
gated repositories and could not be accessed, and the original 3,300-prompt suite was not
available for this run, so prompt distribution changes alongside the generator.

Run notebook 09 first: this notebook loads the weights it saves.

## 1. Config

In [ ]:
# CONFIG
REPO_DIR    = '/content/drive/MyDrive/kazakh-ai-text-detection'
RESULTS_DIR = '/content/drive/MyDrive/kazakh-ai-text-detection/results_r2_crossgen'
WEIGHTS_DIR = '/content/drive/MyDrive/kazakh-ai-text-detection/results_r2_seeds/weights'

BACKEND = 'jax'

# ---- generator (UNSEEN -- must NOT be the one that built the benchmark) ----
#
GEN_MODEL_ID   = 'mistralai/Mistral-7B-Instruct-v0.3'

# Llama-3.1 derivatives are usually gated: accept the terms on the model page
# while logged in, then paste a read token here (or leave blank if not gated).
HF_TOKEN       = ''

# On a 40 GB A100 an 8B model runs in fp16 with room to spare -- no quantisation,
# so generation is faster and output quality is not altered by 4-bit rounding.
# Set True only on a 16 GB card.
LOAD_IN_4BIT   = False

USE_API        = False  # alternative: OpenAI-compatible endpoint, no GPU needed
API_MODEL      = ''
API_BASE_URL   = ''
N_GENERATE     = 345    # matches the human test-class size
MAX_NEW_TOKENS = 400
TEMPERATURE    = 0.8    # same decoding settings as the original corpus
TOP_P          = 0.95
GEN_BATCH      = 16     # A100; drop to 4-8 on a smaller card
SEED           = 42

ORIGINAL_PROMPT_FILE = ''   # <-- STRONGLY PREFERRED: your original 3,300-prompt file

# ---- classifiers ----
EVAL_MODELS = {
    'mDeBERTa-v3 (base)': ('deberta_v3',  'deberta_v3_base_multi'),
    # 'XLM-R (base)':     ('xlm_roberta', 'xlm_roberta_base_multi'),  # enable if 09 saved its weights
}
CLF_SEEDS        = [42, 1, 2]
TRAIN_IF_MISSING = False    # keep False: train-on-the-fly produces a model not in your paper

SEQ_LEN = 256; EPOCHS = 5; EFFECTIVE_BATCH = 16
LEARNING_RATE = 2e-5; WEIGHT_DECAY = 0.01; PATIENCE = 2

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'transformers', 'accelerate', 'bitsandbytes', 'sentencepiece',
                'keras>=3.3', 'keras-hub', 'scikit-learn', 'pandas', 'openai'], check=False)

from google.colab import drive
drive.mount('/content/drive')

try:
    import json as _json
    from pathlib import Path as _Path
    _p = _Path('/content/drive/MyDrive/r2_config.json')
    if _p.exists():
        _cfg = _json.load(open(_p))
        REPO_DIR = _cfg['REPO_DIR']
        RESULTS_DIR = _cfg.get('RESULTS_CROSSGEN_DIR', RESULTS_DIR)
        WEIGHTS_DIR = _cfg.get('WEIGHTS_DIR', WEIGHTS_DIR)
        print('Paths loaded from r2_config.json')
except Exception as _e:
    print('Could not load r2_config.json (%s)' % type(_e).__name__)
print('REPO_DIR   =', REPO_DIR)
print('WEIGHTS_DIR=', WEIGHTS_DIR)


## 2. Imports and environment

In [ ]:
import os, json, gc, time, random, warnings, itertools
# Backend must be chosen BEFORE keras is imported.
os.environ['KERAS_BACKEND'] = BACKEND
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if BACKEND == 'jax':
    # JAX preallocates ~75% of VRAM by default, which starves the TF ops used
    # for tokenisation and makes OOM look like a model-size problem.
    os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

from pathlib import Path
import numpy as np, pandas as pd
import tensorflow as tf
import keras
try:
    import keras_nlp as KNLP
except ImportError:
    import keras_hub as KNLP
from scipy.special import softmax
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings('ignore')

# GPU ownership.
#
_gpus = tf.config.list_physical_devices('GPU')
if BACKEND == 'jax' and _gpus:
    try:
        tf.config.set_visible_devices([], 'GPU')
        print('TensorFlow: GPU hidden -- JAX owns the device (tokenisation stays on CPU)')
    except Exception as e:
        print('WARNING: could not hide the GPU from TensorFlow:', e)
        print('Restart the runtime and run this cell before any other TF op.')
else:
    for _d in _gpus:
        try:
            tf.config.experimental.set_memory_growth(_d, True)
        except Exception:
            pass

print(f'TF {tf.__version__} | Keras {keras.__version__} | backend {keras.backend.backend()}')
print('physical GPU:', [d.name for d in _gpus] or 'NONE')

# What card did Colab actually give us, and how much of it is free right now?
try:
    import subprocess as _sp
    _o = _sp.run(['nvidia-smi',
                  '--query-gpu=name,memory.total,memory.used,memory.free',
                  '--format=csv,noheader'], capture_output=True, text=True)
    if _o.returncode == 0 and _o.stdout.strip():
        print('nvidia-smi:', _o.stdout.strip())
        _free = int(_o.stdout.split(',')[-1].strip().split()[0])
        if _free < 9000:
            print(f'WARNING: only ~{_free} MiB free. mDeBERTa-v3 base full fine-tuning needs')
            print('roughly 9-10 GB (278M params x 4 copies for AdamW, plus activations).')
            print('If a previous cell is still holding memory, restart the runtime.')
except Exception:
    pass

if _gpus:
    keras.mixed_precision.set_global_policy('mixed_float16')
    print('mixed precision: mixed_float16')

ID2LABEL = {0: 'Human', 1: 'AI-Generated', 2: 'AI-Obfuscated'}
LABELS = [0, 1, 2]

_REG = {
    'xlm_roberta': ('XLMRobertaClassifier', 'XLMRobertaPreprocessor'),
    'deberta_v3':  ('DebertaV3Classifier',  'DebertaV3Preprocessor'),
    'bert':        ('BertClassifier',       'BertPreprocessor'),
    'distil_bert': ('DistilBertClassifier', 'DistilBertPreprocessor'),
}

def slug(s):
    return s.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')

def load_split(name):
    d = pd.DataFrame(json.load(open(REPO / f'data/cleaned/{name}_cleaned.json', encoding='utf-8')))
    d['text'] = d['text'].astype(str)
    d['label'] = d['label'].astype(int)
    return d

REPO = Path(REPO_DIR); OUT = Path(RESULTS_DIR); OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

train, dev, test = load_split('train'), load_split('dev'), load_split('test')
test['n_words'] = test['text'].str.split().str.len()
human_test = test[test.label == 0].reset_index(drop=True)
print('human test documents held fixed:', len(human_test),
      '| median length', human_test.n_words.median(), 'words')

# ---- PREFLIGHT: do the weights exist? (before any 1 GB download) ----
wd = Path(WEIGHTS_DIR)
print('\nWEIGHTS_DIR:', wd, '| exists:', wd.is_dir())
present = sorted(p.name for p in wd.glob('*.weights.h5')) if wd.is_dir() else []
print('files present:', present or 'NONE')

missing = []
for name in EVAL_MODELS:
    for sd in CLF_SEEDS:
        f = f'{slug(name)}__seed{sd}.weights.h5'
        if f not in present:
            missing.append(f)
if missing:
    print('\nMISSING:', missing)
    if not TRAIN_IF_MISSING:
        raise FileNotFoundError(
            'Required weight files are missing and TRAIN_IF_MISSING is False.\n'
            'Run notebook 09 first -- it produces exactly these files.\n'
            'If notebook 09 saved them under different names, fix WEIGHTS_DIR or the keys of '
            'EVAL_MODELS so that slug(name) matches the filenames listed above. '
            'Do not flip TRAIN_IF_MISSING to work around a naming mismatch: a model trained here '
            'is not the model your manuscript reports.')
    print('TRAIN_IF_MISSING is True -- missing seeds will be trained here. '
          'Disclose this in the paper.')
else:
    print('\npreflight OK: weights present for seeds', CLF_SEEDS)


## 3. Load data

In [ ]:
# Pre-tokenisation.
#
_PREPROC = {}

def get_preprocessor(family, preset, force_new=False):
    key = (family, preset)
    if key not in _PREPROC or force_new:
        _PREPROC[key] = getattr(KNLP.models, _REG[family][1]).from_preset(
            preset, sequence_length=SEQ_LEN)
    return _PREPROC[key]

def encode(texts, family, preset, chunk=256, name=''):
    texts = [str(t) for t in texts]
    parts = None
    for attempt in (0, 1):
        try:
            pre = get_preprocessor(family, preset, force_new=bool(attempt))
            parts = []
            with tf.device('/CPU:0'):
                for i in range(0, len(texts), chunk):
                    o = pre(tf.constant(texts[i:i + chunk], dtype=tf.string))
                    parts.append({k: np.asarray(v) for k, v in dict(o).items()})
            break
        except tf.errors.NotFoundError:
            if attempt:
                raise
            print('  SentencePiece resource invalidated -- rebuilding preprocessor, retrying')
    enc = {k: np.concatenate([p[k] for p in parts], axis=0) for k in parts[0]}
    if name:
        print(f'  encoded {name}: {len(texts)} docs -> {tuple(next(iter(enc.values())).shape)}')
    return enc

def build_clf(family, preset, num_classes=3):
    # preprocessor=None -> the model takes token ids, not raw strings.
    try:
        return getattr(KNLP.models, _REG[family][0]).from_preset(
            preset, num_classes=num_classes, preprocessor=None)
    except Exception as e:
        raise RuntimeError(
            f'Could not load preset {preset!r} ({type(e).__name__}: {e}). '
            'Presets download from Kaggle -- check network access, and set '
            'KAGGLE_USERNAME / KAGGLE_KEY if prompted.') from e

def make_ds(X, y=None, batch=16, shuffle_seed=None):
    n = len(next(iter(X.values())))
    ds = tf.data.Dataset.from_tensor_slices((X, y) if y is not None else X)
    if shuffle_seed is not None:
        ds = ds.shuffle(n, seed=shuffle_seed, reshuffle_each_iteration=True)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)


## 4. Classifier helpers (tokenise, build, train, predict)

In [ ]:
# Memory management.
#
MICRO_BATCHES_PER_MODEL = {
    'mDeBERTa-v3 (base)': [16, 8],
    'XLM-R (base)':       [16, 8],
    'mBERT (cased)':      [16, 8],
    'DistilmBERT':        [16, 8],
}
DEFAULT_MICRO_BATCHES = [16, 8, 4]

# A run at micro-batch 16 uses no accumulation at all; anything smaller does.
# The notebook reports which was used per seed so you can state it in the paper.

def gpu_free_mib():
    try:
        import subprocess
        o = subprocess.run(['nvidia-smi', '--query-gpu=memory.free',
                            '--format=csv,noheader,nounits'],
                           capture_output=True, text=True)
        return int(o.stdout.strip().split('\n')[0]) if o.returncode == 0 else None
    except Exception:
        return None

def hard_reset(*objs):
    # Drop references, clear the Keras session, and clear JAX's compilation
    # caches. Without the JAX step, compiled executables keep device buffers
    # alive across seeds.
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    try:
        keras.backend.clear_session()
    except Exception:
        pass
    if BACKEND == 'jax':
        try:
            import jax
            jax.clear_caches()
        except Exception:
            pass
    gc.collect()

def is_oom(msg):
    m = str(msg)
    return ('RESOURCE_EXHAUSTED' in m or 'ResourceExhausted' in m
            or 'out of memory' in m.lower() or 'Out of memory' in m)

def compile_clf(clf, micro):
    assert EFFECTIVE_BATCH % micro == 0, 'EFFECTIVE_BATCH must divide by the micro-batch'
    accum = EFFECTIVE_BATCH // micro
    kw = {'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY}
    if accum > 1:
        try:
            opt = keras.optimizers.AdamW(gradient_accumulation_steps=accum, **kw)
        except TypeError:
            print(f'  NOTE: this Keras build has no gradient_accumulation_steps; running at '
                  f'true batch {micro} instead of an effective {EFFECTIVE_BATCH}. '
                  f'That is a protocol difference -- record it in the paper.')
            opt = keras.optimizers.AdamW(**kw)
    else:
        opt = keras.optimizers.AdamW(**kw)
    clf.compile(optimizer=opt,
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])
    return accum

def fit_with_backoff(base, family, preset, seed, X_train, y_train, X_dev, y_dev, class_weight):
    micros = MICRO_BATCHES_PER_MODEL.get(base, DEFAULT_MICRO_BATCHES)
    last_msg = ''
    for micro in micros:
        clf = None
        try:
            hard_reset()
            keras.utils.set_random_seed(seed)
            clf = build_clf(family, preset)
            accum = compile_clf(clf, micro)
            free = gpu_free_mib()
            print(f'  training: micro-batch {micro} x accum {accum} = effective '
                  f'{EFFECTIVE_BATCH}' + (f'  |  {free} MiB free' if free else ''))
            hist = clf.fit(
                make_ds(X_train, y_train, batch=micro, shuffle_seed=seed),
                validation_data=make_ds(X_dev, y_dev, batch=micro),
                epochs=EPOCHS, class_weight=class_weight,
                callbacks=[keras.callbacks.EarlyStopping(
                    monitor='val_loss', patience=PATIENCE, restore_best_weights=True)],
                verbose=1)
            return clf, hist.history, micro
        except Exception as e:
            # Store a STRING, not the exception: keeping `e` keeps its traceback,
            # which keeps the frames, which keep this dead model on the GPU.
            last_msg = f'{type(e).__name__}: {e}'
            tb_free = is_oom(last_msg)
            clf = None
            hard_reset()
            if not tb_free:
                raise RuntimeError(last_msg)
            print(f'  OOM at micro-batch {micro} -- backing off '
                  f'({gpu_free_mib()} MiB free after cleanup)')
    raise MemoryError(
        f'Out of memory even at micro-batch {micros[-1]} for {base}.\n'
        f'  Free VRAM right now: {gpu_free_mib()} MiB.\n'
        f'  If an earlier seed succeeded and this one did not, memory is not being '
        f'released -- RESTART THE RUNTIME and re-run; finished seeds are skipped.\n'
        f'  Otherwise: Runtime > Change runtime type -> L4 or A100, or train the '
        f'smaller encoders first.\n'
        f'Last error: {last_msg}')

def predict_with_backoff(clf, X, start=None):
    for b in ([start] if start else []) + [64, 32, 16, 8]:
        try:
            return softmax(np.asarray(clf.predict(make_ds(X, batch=b), verbose=0)), axis=1)
        except Exception as e:
            if not is_oom(f'{type(e).__name__}: {e}'):
                raise
            print(f'  OOM predicting at batch {b} -- backing off')
            gc.collect()
    raise MemoryError('Out of memory predicting even at batch 8.')

def save_model_only_weights(clf, family, preset, path):
    # A compiled model's weight file carries AdamW's moment estimates too
    # (~4x the size). Copy to host, free the GPU, then save from a fresh
    # uncompiled model so the file holds parameters only.
    w = clf.get_weights()
    hard_reset(clf)
    tmp = build_clf(family, preset)
    tmp.set_weights(w)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp.save_weights(str(path))
    hard_reset(tmp)
    del w
    gc.collect()


## 5. Prompt suite

In [ ]:
TASK_TEMPLATES = {
    'news_brief':  'Төмендегі тақырып бойынша қысқа жаңалық хабарламасын жазыңыз: {topic}',
    'short_essay': 'Төмендегі тақырып бойынша қысқа эссе жазыңыз: {topic}',
    'explanation': 'Төмендегі тақырыпты түсіндіріп жазыңыз: {topic}',
    'opinion':     'Төмендегі тақырып бойынша өз пікіріңізді білдіріңіз: {topic}',
    'description': 'Төмендегі тақырыпқа сипаттама жазыңыз: {topic}',
    'short_story': 'Төмендегі тақырып бойынша қысқа әңгіме жазыңыз: {topic}',
}
PLACEHOLDER_TOPICS = [
    'Қазақтың дәстүрлі музыкасы', 'Наурыз мейрамы', 'Қазақ халқының ұлттық тағамдары',
    'Көшпелі өмір салты', 'Қазақ киіз үйі', 'Ұлттық спорт түрлері',
    'Қазақ ауыз әдебиеті', 'Домбыра және күй өнері', 'Қазақстанның табиғаты',
    'Ұлы Жібек жолы', 'Қазақ тілінің дамуы', 'Ұлттық қолөнер',
    'Қазақстандағы білім беру', 'Абай Құнанбаевтың шығармашылығы',
    'Қазақ халқының салт-дәстүрлері', 'Қазақстанның қалалары',
    'Жылқы малын өсіру дәстүрі', 'Ұлттық киім үлгілері',
    'Қазақ айтыс өнері', 'Қазақстандағы заманауи мәдениет',
]

prompts = []
if ORIGINAL_PROMPT_FILE and Path(ORIGINAL_PROMPT_FILE).exists():
    p = Path(ORIGINAL_PROMPT_FILE)
    if p.suffix == '.json':
        obj = json.load(open(p, encoding='utf-8'))
        raw = obj if isinstance(obj, list) else obj.get('prompts', [])
        prompts = [x if isinstance(x, str) else x.get('prompt', '') for x in raw]
    elif p.suffix == '.jsonl':
        prompts = [json.loads(l).get('prompt', '') for l in open(p, encoding='utf-8')]
    else:
        prompts = [l.strip() for l in open(p, encoding='utf-8') if l.strip()]
    prompts = [x for x in prompts if x]
    print(f'Using the ORIGINAL prompt suite: {len(prompts)} prompts. '
          'The generator is the only changed variable.')
else:
    print('*** WARNING: original prompt file not found -- using PLACEHOLDER topics. ***')
    print('*** Prompt distribution becomes a second changed variable. Disclose it. ***')
    for topic in PLACEHOLDER_TOPICS:
        for tmpl in TASK_TEMPLATES.values():
            prompts.append(tmpl.format(topic=topic))

rng.shuffle(prompts)
prompts = (prompts[:N_GENERATE] if len(prompts) >= N_GENERATE
           else [prompts[i % len(prompts)] for i in range(N_GENERATE)])
print('prompts to run:', len(prompts), '| example:', prompts[0])


## 6. Resumable generation with the unseen generator

In [ ]:
# ---------------- resumable generation ----------------
GEN_PATH = OUT / 'ood_generations.jsonl'

def load_done(path):
    done = {}
    if Path(path).exists():
        for line in open(path, encoding='utf-8'):
            try:
                r = json.loads(line); done[r['gen_id']] = r
            except Exception:
                continue
    return done

items = [{'gen_id': f'ood_{i:05d}', 'prompt': p} for i, p in enumerate(prompts)]
done = load_done(GEN_PATH)
todo = [it for it in items if it['gen_id'] not in done]
print(f'{len(done)} done, {len(todo)} remaining')

if todo:
    if USE_API:
        if not API_MODEL:
            raise ValueError('USE_API is True but API_MODEL is empty.')
        from openai import OpenAI
        client = OpenAI(base_url=API_BASE_URL or None)
        with open(GEN_PATH, 'a', encoding='utf-8') as fout:
            for k, it in enumerate(todo, 1):
                r = client.chat.completions.create(
                    model=API_MODEL, messages=[{'role': 'user', 'content': it['prompt']}],
                    temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS)
                it = dict(it); it['text'] = r.choices[0].message.content.strip()
                it['generator'] = API_MODEL
                it['finish_reason'] = r.choices[0].finish_reason
                fout.write(json.dumps(it, ensure_ascii=False) + '\n'); fout.flush()
                if k % 25 == 0:
                    print(f'  {k}/{len(todo)}', flush=True)
    else:
        if not GEN_MODEL_ID:
            raise ValueError(
                'GEN_MODEL_ID is empty and USE_API is False. Set GEN_MODEL_ID to a VERIFIED '
                'Hugging Face repo id of a generator that was NOT used to build the benchmark, '
                'or set USE_API=True with API_MODEL. Do not guess a repo id.')
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        if HF_TOKEN:
            from huggingface_hub import login
            login(token=HF_TOKEN)
            print('logged in to Hugging Face')
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                   bnb_4bit_compute_dtype=torch.float16,
                                   bnb_4bit_use_double_quant=True) if LOAD_IN_4BIT else None
        try:
            gtok = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
        except Exception as e:
            raise RuntimeError(
                f'Could not load {GEN_MODEL_ID!r} ({type(e).__name__}). '
                'If this is a gated repo: open its page on huggingface.co while logged in, '
                'accept the licence, create a READ access token, and set HF_TOKEN above.') from e
        if gtok.pad_token is None:
            gtok.pad_token = gtok.eos_token
        gtok.padding_side = 'left'
        gmodel = AutoModelForCausalLM.from_pretrained(
            GEN_MODEL_ID, quantization_config=quant, device_map='auto',
            torch_dtype=torch.float16)
        gmodel.eval(); torch.manual_seed(SEED)
        print(f'generator loaded: {GEN_MODEL_ID}'
              f'{"  (4-bit)" if LOAD_IN_4BIT else "  (fp16)"}')

        def chat(p):
            try:
                return gtok.apply_chat_template([{'role': 'user', 'content': p}],
                                                tokenize=False, add_generation_prompt=True)
            except Exception:
                return p

        with open(GEN_PATH, 'a', encoding='utf-8') as fout:
            for s in range(0, len(todo), GEN_BATCH):
                chunk = todo[s:s + GEN_BATCH]
                enc = gtok([chat(it['prompt']) for it in chunk], return_tensors='pt',
                           padding=True, truncation=True, max_length=512).to(gmodel.device)
                with torch.no_grad():
                    out = gmodel.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                          temperature=TEMPERATURE, top_p=TOP_P,
                                          pad_token_id=gtok.pad_token_id)
                newtok = out[:, enc['input_ids'].shape[1]:]
                texts = gtok.batch_decode(newtok, skip_special_tokens=True)
                for it, g, row in zip(chunk, texts, newtok):
                    it = dict(it); it['text'] = g.strip(); it['generator'] = GEN_MODEL_ID
                    # flag length-capped outputs so truncation is not mistaken for brevity
                    it['hit_token_cap'] = bool(int((row != gtok.pad_token_id).sum())
                                               >= MAX_NEW_TOKENS)
                    fout.write(json.dumps(it, ensure_ascii=False) + '\n')
                fout.flush()
                print(f'  {min(s+GEN_BATCH, len(todo))}/{len(todo)}', flush=True)
        del gmodel; gc.collect()
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

done = load_done(GEN_PATH)
print('total generated:', len(done))


## 7. Quality control and the out-of-domain evaluation set

In [ ]:
# ---------------- quality control ----------------
ood = pd.DataFrame(list(done.values()))
ood['text'] = ood['text'].astype(str).str.strip()
ood['n_words'] = ood['text'].str.split().str.len()

def cyr_ratio(t):
    return sum('Ѐ' <= c <= 'ӿ' for c in t) / len(t) if t else 0.0
ood['cyr_ratio'] = ood['text'].map(cyr_ratio)

before = len(ood)
ood = ood[(ood.n_words >= 10) & (ood.cyr_ratio >= 0.5)].reset_index(drop=True)
print(f'kept {len(ood)}/{before} after quality filtering')
if 'hit_token_cap' in ood:
    capped = int(ood['hit_token_cap'].sum())
    print(f'{capped} generations hit the {MAX_NEW_TOKENS}-token cap '
          f'-- their length is censored, so do not report their mean length as a property '
          f'of the generator.')

print('\nLength comparison (a large mismatch is a confound to disclose):')
print('  human test   median words:', human_test.n_words.median())
print('  in-domain AI median words:', test[test.label == 1].n_words.median())
print('  OOD AI       median words:', ood.n_words.median())

ood_eval = pd.concat([
    human_test[['text']].assign(label=0, origin='human_test'),
    ood[['text']].assign(label=1, origin='ood_generator'),
], ignore_index=True)
ood_eval.to_json(OUT / 'ood_eval_set.jsonl', orient='records', lines=True, force_ascii=False)
print('\nOOD evaluation set:', len(ood_eval), dict(ood_eval.label.value_counts()))


## 8. Zero-shot evaluation under the binary mapping

In [ ]:
# ---------------- zero-shot evaluation ----------------
y_train = train['label'].to_numpy('int32'); y_dev = dev['label'].to_numpy('int32')
y_test  = test['label'].to_numpy(); y_ood = ood_eval['label'].to_numpy()
cw = compute_class_weight('balanced', classes=np.array(LABELS), y=y_train)
class_weight = {int(k): float(v) for k, v in zip(LABELS, cw)}

def to_binary(p):    # operational mapping from Section III-B: {1,2} -> machine
    return (np.asarray(p) >= 1).astype(int)

rows = []
for name, (family, preset) in EVAL_MODELS.items():
    print(f'\n===== {name} =====')
    print('tokenising')
    X_ood  = encode(ood_eval['text'], family, preset, name='OOD set')
    X_test = encode(test['text'],     family, preset, name='in-domain test')
    X_tr = X_dv = None

    acc_ood = acc_id = None
    for sd in CLF_SEEDS:
        wp = Path(WEIGHTS_DIR) / f'{slug(name)}__seed{sd}.weights.h5'
        keras.backend.clear_session(); gc.collect()
        if wp.exists():
            clf = build_clf(family, preset)
            compile_clf(clf, EFFECTIVE_BATCH)
            clf.load_weights(str(wp))
            print(f'  loaded {wp.name}')
        elif TRAIN_IF_MISSING:
            if X_tr is None:
                X_tr = encode(train['text'], family, preset, name='train')
                X_dv = encode(dev['text'],   family, preset, name='dev')
            clf, _, _ = fit_with_backoff(name, family, preset, sd, X_tr, y_train,
                                         X_dv, y_dev, class_weight)
        else:
            raise FileNotFoundError(f'{wp} missing -- run notebook 09.')

        acc_ood = (lambda p: p if acc_ood is None else acc_ood + p)(predict_with_backoff(clf, X_ood))
        acc_id  = (lambda p: p if acc_id  is None else acc_id  + p)(predict_with_backoff(clf, X_test))
        del clf; gc.collect()

    p_ood, p_id = acc_ood / len(CLF_SEEDS), acc_id / len(CLF_SEEDS)
    np.save(OUT / f'probs_ood_{slug(name)}.npy', p_ood)
    yp_ood, yp_id = p_ood.argmax(1), p_id.argmax(1)

    # anchor: in-domain must reproduce the manuscript before the OOD number means anything
    id_macro = f1_score(y_test, yp_id, average='macro')
    print(f'  in-domain macro-F1 (anchor) = {id_macro:.4f}')
    if id_macro < 0.70:
        print('  *** Far below the published figure. The weights, preset or label mapping is '
              'wrong. Do not interpret the OOD numbers until this is resolved. ***')

    b_ood_t, b_ood_p = to_binary(y_ood), to_binary(yp_ood)
    b_id_t,  b_id_p  = to_binary(y_test), to_binary(yp_id)
    rows.append({
        'model': name,
        'macro_f1_indomain_3way': round(id_macro, 4),
        'binary_f1_indomain': round(f1_score(b_id_t, b_id_p, average='macro'), 4),
        'binary_acc_indomain': round(accuracy_score(b_id_t, b_id_p), 4),
        'binary_f1_ood': round(f1_score(b_ood_t, b_ood_p, average='macro'), 4),
        'binary_acc_ood': round(accuracy_score(b_ood_t, b_ood_p), 4),
        'binary_f1_drop': round(f1_score(b_ood_t, b_ood_p, average='macro')
                                - f1_score(b_id_t, b_id_p, average='macro'), 4),
        'ood_recall_machine': round(float((yp_ood[y_ood == 1] >= 1).mean()), 4),
        'ood_recall_human': round(float((yp_ood[y_ood == 0] == 0).mean()), 4),
        'ood_pct_machine_called_obfuscated': round(float((yp_ood[y_ood == 1] == 2).mean()), 4),
    })
    print(classification_report(b_ood_t, b_ood_p,
                                target_names=['human', 'machine'], digits=4))
    del X_ood, X_test; gc.collect()

cross = pd.DataFrame(rows)
cross.to_csv(OUT / 'table_crossgenerator.csv', index=False)
print(cross.to_string(index=False))
